# OncoVision: train the breast MRI models on MAMA-MIA

Trains on 1,506 breast cancer DCE-MRI scans with expert 3D tumor outlines:
- a **3D tumor finder** for whole breast MRI scans,
- a **2D tumor finder** for single MRI images,
- plus example slices so the app learns to recognise breast MRI.

**How to run (important, so nothing gets lost):**
1. Right panel: **Accelerator → GPU T4 x2** (or P100) and **Internet → On**.
2. Top right: **Save Version → Save & Run All (Commit) → Save**. Do *not* use the normal Run All.
3. Close the tab if you like. It runs in the background for about 4 hours.
4. Later open the notebook page → **Output** tab → download `oncovision_breast_mri.zip` and send it to Claude together with the printed `SCORES` line.

Data: MAMA-MIA (Garrucho et al., Scientific Data 2025) via the Hugging Face redistribution YongchengYAO/MAMA-MIA-Lite, **CC BY-NC 4.0 (non-commercial use only)**. Each scan is read straight out of the remote archive, so nothing large is downloaded.

In [ ]:
!git clone -q -b vaishnavi/optimistic-noether-034odi https://github.com/Vaishnavi220506/Medical.git /kaggle/working/oncovision
%cd /kaggle/working/oncovision
!pip install -q -r requirements.txt huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings!")


## 1. Prepare and train the 3D model (about 2-3 hours)
160 patients are held out: 60 for validation, 100 for the final test.

In [ ]:
!python -m oncovision.training.train_breast_mri3d --prepared-dir /tmp/breast3d --epochs 60 --samples-per-epoch 800 \
    --patch 96 --batch-size 4 --val-cases 60 --test-cases 100 --val-every 5 --workers 4 --out weights/breast_mri/segmenter3d.pt


## 2. 2D model for single MRI images (about 1 hour)

In [ ]:
!python -m oncovision.training.export_slices --prepared-dir /tmp/breast3d --out /tmp/breast2d --split train --lesion-slices 5 --empty-slices 3 --router 800
!python -m oncovision.training.export_slices --prepared-dir /tmp/breast3d --out /tmp/breast2d_test --split test --lesion-slices 5 --empty-slices 3
!python -m oncovision.training.train_segmenter --profile breast_mri --format maskfolder --data /tmp/breast2d/images --masks /tmp/breast2d/masks \
    --epochs 20 --batch-size 16 --workers 4 --out weights/breast_mri/segmenter.pt
!python -m oncovision.training.evaluate segmenter --profile breast_mri --format maskfolder --data /tmp/breast2d_test/images \
    --masks /tmp/breast2d_test/masks --weights weights/breast_mri/segmenter.pt --out /kaggle/working/breast2d_test


## 3. Scores and package (includes 800 example slices for the scan-type router)

In [ ]:
import torch, json, glob
scores = {}
for path in sorted(glob.glob("weights/breast_mri/*.pt")):
    meta = torch.load(path, map_location="cpu", weights_only=True)["meta"]
    scores[path.split("/")[-1]] = {k: {kk: vv for kk, vv in v.items() if kk != "by_threshold"} for k, v in meta.get("metrics", {}).items()}
for path in glob.glob("/kaggle/working/*2d_test/metrics.json"):
    scores[path.split("/")[-2]] = {k: v for k, v in json.load(open(path)).items() if not isinstance(v, (list, dict)) or k == "dice"}
print("SCORES (send these to Claude):", json.dumps(scores))
!python scripts/export_weights.py --src weights/breast_mri --dst export_weights/breast_mri
!mkdir -p export_weights/router_breast_mri && cp /tmp/breast2d/router/*.png export_weights/router_breast_mri/
!cd /kaggle/working/oncovision/export_weights && rm -f /kaggle/working/oncovision_breast_mri.zip && zip -qr /kaggle/working/oncovision_breast_mri.zip breast_mri router_breast_mri && ls -la /kaggle/working/oncovision_breast_mri.zip
